<a href="https://colab.research.google.com/github/itsriddhima/Evidence-Guard/blob/main/evidenceguard_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!mkdir -p data outputs
!curl -L -o data/response.jsonl https://raw.githubusercontent.com/ParticleMedia/RAGTruth/main/dataset/response.jsonl
!curl -L -o data/source_info.jsonl https://raw.githubusercontent.com/ParticleMedia/RAGTruth/main/dataset/source_info.jsonl
!ls -lh data/

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 20.4M  100 20.4M    0     0  37.9M      0 --:--:-- --:--:-- --:--:-- 37.9M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 14.4M  100 14.4M    0     0  26.8M      0 --:--:-- --:--:-- --:--:-- 26.8M
total 35M
-rw-r--r-- 1 root root 21M Nov 25 13:46 response.jsonl
-rw-r--r-- 1 root root 15M Nov 25 13:46 source_info.jsonl


In [ ]:
!pip install -q sentence-transformers transformers accelerate pyarrow

In [ ]:
import json
from pathlib import Path
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

DATA_DIR = Path("data")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

def load_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def to_text(val) -> str:
    if val is None:
        return ""
    if isinstance(val, str):
        return val
    return json.dumps(val, ensure_ascii=False)

def parse_labels(labels) -> tuple[int, bool, bool, bool]:
    if not isinstance(labels, list):
        return 0, False, False, False
    conflict = False
    baseless = False
    implicit_true = False
    for item in labels:
        if not isinstance(item, dict):
            continue
        t = str(item.get("label_type", "")).strip()
        if t == "Evident Conflict":
            conflict = True
        elif t == "Evident Baseless Info":
            baseless = True
        elif t == "implicit_true":
            implicit_true = True
    is_pos = 1 if (conflict or baseless) else 0
    return is_pos, conflict, baseless, implicit_true

def prepare_data():
    df_resp = pd.DataFrame(load_jsonl(DATA_DIR / "response.jsonl"))
    df_src = pd.DataFrame(load_jsonl(DATA_DIR / "source_info.jsonl"))[["source_id", "task_type", "source_info"]]

    df = df_resp.merge(df_src, on="source_id", how="inner")
    df["source_text"] = df["source_info"].apply(to_text)
    df["response_text"] = df["response"].apply(to_text)

    total_raw = len(df)
    df = df[df["source_text"].str.strip().ne("") & df["response_text"].str.strip().ne("")].copy()
    removed = total_raw - len(df)

    parsed = df["labels"].apply(parse_labels)
    df["label"] = [x[0] for x in parsed]
    df["has_conflict"] = [x[1] for x in parsed]
    df["has_baseless"] = [x[2] for x in parsed]
    df["only_implicit_true"] = [(x[0] == 0 and x[3]) for x in parsed]

    mixed_sources = int((df.groupby("source_id")["split"].nunique() > 1).sum()) if "split" in df.columns else 0

    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
    tv_idx, test_idx = next(gss1.split(df, df["label"], groups=df["source_id"]))
    df_tv = df.iloc[tv_idx].copy()
    df_test = df.iloc[test_idx].copy()

    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)
    tr_idx, val_idx = next(gss2.split(df_tv, df_tv["label"], groups=df_tv["source_id"]))
    df_train = df_tv.iloc[tr_idx].copy()
    df_val = df_tv.iloc[val_idx].copy()

    df_train["eval_split"] = "train"
    df_val["eval_split"] = "val"
    df_test["eval_split"] = "test"

    out_df = pd.concat([df_train, df_val, df_test], ignore_index=True)
    out_df.to_parquet(OUT_DIR / "prepared.parquet", index=False)

    print("Merged:", total_raw, "| Removed empty:", removed, "| Mixed source_ids in raw split:", mixed_sources)
    print("Splits (train/val/test):", len(df_train), len(df_val), len(df_test))
    for name, part in [("train", df_train), ("val", df_val), ("test", df_test)]:
        print(f"  {name}: n={len(part)}, pos={part['label'].sum()} ({part['label'].mean()*100:.2f}%)")
    print("Conflict responses:", int(df["has_conflict"].sum()), "| Baseless responses:", int(df["has_baseless"].sum()))
    print("Only implicit_true (kept as negative=0):", int(df["only_implicit_true"].sum()))

prepare_data()


ArrowTypeError: ("Expected bytes, got a 'dict' object", 'Conversion failed for column source_info with type object')

In [ ]:
import json
from pathlib import Path
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

DATA_DIR = Path("data")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

def load_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def to_text(val) -> str:
    if val is None:
        return ""
    if isinstance(val, str):
        return val
    return json.dumps(val, ensure_ascii=False)

def parse_labels(labels) -> tuple[int, bool, bool, bool]:
    if not isinstance(labels, list):
        return 0, False, False, False
    conflict = False
    baseless = False
    implicit_true = False
    for item in labels:
        if not isinstance(item, dict):
            continue
        t = str(item.get("label_type", "")).strip()
        if t == "Evident Conflict":
            conflict = True
        elif t == "Evident Baseless Info":
            baseless = True
        elif t == "implicit_true":
            implicit_true = True
    is_pos = 1 if (conflict or baseless) else 0
    return is_pos, conflict, baseless, implicit_true

def prepare_data():
    df_resp = pd.DataFrame(load_jsonl(DATA_DIR / "response.jsonl"))
    df_src = pd.DataFrame(load_jsonl(DATA_DIR / "source_info.jsonl"))[["source_id", "task_type", "source_info"]]

    df = df_resp.merge(df_src, on="source_id", how="inner")
    df["source_text"] = df["source_info"].apply(to_text)
    df["response_text"] = df["response"].apply(to_text)

    total_raw = len(df)
    df = df[df["source_text"].str.strip().ne("") & df["response_text"].str.strip().ne("")].copy()
    removed = total_raw - len(df)

    parsed = df["labels"].apply(parse_labels)
    df["label"] = [x[0] for x in parsed]
    df["has_conflict"] = [x[1] for x in parsed]
    df["has_baseless"] = [x[2] for x in parsed]
    df["only_implicit_true"] = [(x[0] == 0 and x[3]) for x in parsed]

    mixed_sources = int((df.groupby("source_id")["split"].nunique() > 1).sum()) if "split" in df.columns else 0

    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
    tv_idx, test_idx = next(gss1.split(df, df["label"], groups=df["source_id"]))
    df_tv = df.iloc[tv_idx].copy()
    df_test = df.iloc[test_idx].copy()

    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)
    tr_idx, val_idx = next(gss2.split(df_tv, df_tv["label"], groups=df_tv["source_id"]))
    df_train = df_tv.iloc[tr_idx].copy()
    df_val = df_tv.iloc[val_idx].copy()

    df_train["eval_split"] = "train"
    df_val["eval_split"] = "val"
    df_test["eval_split"] = "test"

    keep_cols = [
        "source_id", "task_type", "eval_split", "label",
        "has_conflict", "has_baseless", "only_implicit_true",
        "source_text", "response_text"
    ]
    if "model" in df.columns:
        keep_cols.append("model")
    if "split" in df.columns:
        keep_cols.append("split")

    out_df = pd.concat([df_train, df_val, df_test], ignore_index=True)[keep_cols]
    out_df.to_parquet(OUT_DIR / "prepared.parquet", index=False)

    print("Merged:", total_raw, "| Removed empty:", removed, "| Mixed source_ids in raw split:", mixed_sources)
    print("Splits (train/val/test):", len(df_train), len(df_val), len(df_test))
    for name, part in [("train", df_train), ("val", df_val), ("test", df_test)]:
        print(f"  {name}: n={len(part)}, pos={part['label'].sum()} ({part['label'].mean()*100:.2f}%)")
    print("Conflict responses:", int(df["has_conflict"].sum()), "| Baseless responses:", int(df["has_baseless"].sum()))
    print("Only implicit_true (kept as negative=0):", int(df["only_implicit_true"].sum()))

prepare_data()

Merged: 17790 | Removed empty: 0 | Mixed source_ids in raw split: 0
Splits (train/val/test): 12450 2670 2670
  train: n=12450, pos=4734 (38.02%)
  val: n=2670, pos=1013 (37.94%)
  test: n=2670, pos=1008 (37.75%)
Conflict responses: 3713 | Baseless responses: 4393
Only implicit_true (kept as negative=0): 0


In [ ]:
import time
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score

MAX_EVAL_SAMPLES = 250
BATCH_SIZE = 32
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

df = pd.read_parquet("outputs/prepared.parquet")
test_df = df[df["eval_split"] == "test"].copy()
if MAX_EVAL_SAMPLES is not None:
    test_df = test_df.head(MAX_EVAL_SAMPLES).copy()

def eval_detector(name: str, y_true: np.ndarray, hall_scores: np.ndarray, elapsed_sec: float):
    roc = roc_auc_score(y_true, hall_scores)
    pr = average_precision_score(y_true, hall_scores)
    thresholds = np.linspace(np.min(hall_scores), np.max(hall_scores), 200)
    best_f1, best_t = 0.0, 0.5
    for t in thresholds:
        preds = (hall_scores >= t).astype(int)
        score = f1_score(y_true, preds, zero_division=0)
        if score > best_f1:
            best_f1, best_t = score, float(t)
    preds_best = (hall_scores >= best_t).astype(int)
    return {
        "model": name,
        "n_test": len(y_true),
        "roc_auc": round(float(roc), 4),
        "pr_auc": round(float(pr), 4),
        "best_f1": round(float(best_f1), 4),
        "precision": round(float(precision_score(y_true, preds_best, zero_division=0)), 4),
        "recall": round(float(recall_score(y_true, preds_best, zero_division=0)), 4),
        "thresh": round(best_t, 4),
        "sec": round(elapsed_sec, 2),
    }

# 1. MiniLM cosine similarity baseline
t0 = time.time()
minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
src_emb = minilm.encode(test_df["source_text"].str.slice(0, 1500).tolist(), batch_size=BATCH_SIZE, convert_to_tensor=True, show_progress_bar=False)
resp_emb = minilm.encode(test_df["response_text"].str.slice(0, 1500).tolist(), batch_size=BATCH_SIZE, convert_to_tensor=True, show_progress_bar=False)
cos_sims = torch.nn.functional.cosine_similarity(src_emb, resp_emb).cpu().numpy()
minilm_hall = 1.0 - cos_sims
minilm_time = time.time() - t0

# 2. DeBERTa NLI cross-encoder
t0 = time.time()
nli_model = CrossEncoder("cross-encoder/nli-deberta-v3-small", device=device)
pairs = list(zip(test_df["source_text"].str.slice(0, 1200).tolist(), test_df["response_text"].str.slice(0, 1200).tolist()))
nli_logits = nli_model.predict(pairs, batch_size=BATCH_SIZE, show_progress_bar=False)
nli_probs = torch.softmax(torch.tensor(nli_logits), dim=-1).numpy()
nli_hall = 1.0 - nli_probs[:, 1]
nli_time = time.time() - t0

# 3. Capped Vectara HHEM
t0 = time.time()
hhem_name = "vectara/hallucination_evaluation_model"
tokenizer = AutoTokenizer.from_pretrained(hhem_name)
hhem_model = AutoModelForSequenceClassification.from_pretrained(hhem_name).to(device)
hhem_model.eval()

hhem_scores = []
src_list = test_df["source_text"].str.slice(0, 1200).tolist()
resp_list = test_df["response_text"].str.slice(0, 1200).tolist()

for i in range(0, len(src_list), BATCH_SIZE):
    batch_texts = [
        f"<pad> {s} </s><s> {r}"
        for s, r in zip(src_list[i:i+BATCH_SIZE], resp_list[i:i+BATCH_SIZE])
    ]
    enc = tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = hhem_model(**enc).logits
        probs = torch.softmax(logits, dim=-1)
        consistent_prob = probs[:, 0].cpu().numpy() if probs.shape[-1] > 1 else torch.sigmoid(logits.squeeze(-1)).cpu().numpy()
        hhem_scores.extend((1.0 - consistent_prob).tolist())

hhem_hall = np.array(hhem_scores)
hhem_time = time.time() - t0

y_test = test_df["label"].to_numpy()
results_df = pd.DataFrame([
    eval_detector("MiniLM (1 - cos_sim)", y_test, minilm_hall, minilm_time),
    eval_detector("DeBERTa-v3-small NLI (1 - P_entail)", y_test, nli_hall, nli_time),
    eval_detector("Vectara HHEM (1 - P_consistent, capped)", y_test, hhem_hall, hhem_time),
])

results_df.to_csv("outputs/benchmark_results.csv", index=False)
print(results_df.to_string(index=False))

Using device: cpu


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

The repository vectara/hallucination_evaluation_model contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/vectara/hallucination_evaluation_model .
 You can inspect the repository content at https://hf.co/vectara/hallucination_evaluation_model.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


[transformers] You are using a model of type `HHEMv2Config` to instantiate a model of type `HHEMv2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.

In [8]:
!pip install -q sentencepiece protobuf

In [9]:
import time
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score

MAX_EVAL_SAMPLES = 250
BATCH_SIZE = 32
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

df = pd.read_parquet("outputs/prepared.parquet")
test_df = df[df["eval_split"] == "test"].copy()
if MAX_EVAL_SAMPLES is not None:
    test_df = test_df.head(MAX_EVAL_SAMPLES).copy()

y_test = test_df["label"].to_numpy()
src_list = test_df["source_text"].str.slice(0, 1200).tolist()
resp_list = test_df["response_text"].str.slice(0, 1200).tolist()

def eval_detector(name: str, y_true: np.ndarray, hall_scores: np.ndarray, elapsed_sec: float):
    roc = roc_auc_score(y_true, hall_scores)
    pr = average_precision_score(y_true, hall_scores)
    thresholds = np.linspace(np.min(hall_scores), np.max(hall_scores), 200)
    best_f1, best_t = 0.0, 0.5
    for t in thresholds:
        preds = (hall_scores >= t).astype(int)
        score = f1_score(y_true, preds, zero_division=0)
        if score > best_f1:
            best_f1, best_t = score, float(t)
    preds_best = (hall_scores >= best_t).astype(int)
    return {
        "model": name,
        "n_test": len(y_true),
        "roc_auc": round(float(roc), 4),
        "pr_auc": round(float(pr), 4),
        "best_f1": round(float(best_f1), 4),
        "precision": round(float(precision_score(y_true, preds_best, zero_division=0)), 4),
        "recall": round(float(recall_score(y_true, preds_best, zero_division=0)), 4),
        "thresh": round(best_t, 4),
        "sec": round(elapsed_sec, 2),
    }

rows = []

# 1. MiniLM cosine similarity baseline
t0 = time.time()
minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
src_emb = minilm.encode(test_df["source_text"].str.slice(0, 1500).tolist(), batch_size=BATCH_SIZE, convert_to_tensor=True, show_progress_bar=False)
resp_emb = minilm.encode(test_df["response_text"].str.slice(0, 1500).tolist(), batch_size=BATCH_SIZE, convert_to_tensor=True, show_progress_bar=False)
cos_sims = torch.nn.functional.cosine_similarity(src_emb, resp_emb).cpu().numpy()
minilm_hall = 1.0 - cos_sims
rows.append(eval_detector("MiniLM (1 - cos_sim)", y_test, minilm_hall, time.time() - t0))
print("Done: MiniLM")

# 2. DeBERTa NLI cross-encoder
t0 = time.time()
nli_model = CrossEncoder("cross-encoder/nli-deberta-v3-small", device=device)
pairs = list(zip(src_list, resp_list))
nli_logits = nli_model.predict(pairs, batch_size=BATCH_SIZE, show_progress_bar=False)
nli_probs = torch.softmax(torch.tensor(nli_logits), dim=-1).numpy()
nli_hall = 1.0 - nli_probs[:, 1]
rows.append(eval_detector("DeBERTa-v3-small NLI (1 - P_entail)", y_test, nli_hall, time.time() - t0))
print("Done: DeBERTa NLI")

# 3. Capped Vectara HHEM (via AutoModel with trust_remote_code=True)
try:
    t0 = time.time()
    hhem_name = "vectara/hallucination_evaluation_model"
    tokenizer = AutoTokenizer.from_pretrained(hhem_name, trust_remote_code=True)
    hhem_model = AutoModel.from_pretrained(hhem_name, trust_remote_code=True).to(device)
    hhem_model.eval()

    hhem_scores = []
    for i in range(0, len(src_list), BATCH_SIZE):
        s_batch = src_list[i:i+BATCH_SIZE]
        r_batch = resp_list[i:i+BATCH_SIZE]
        if hasattr(hhem_model, "predict"):
            batch_pairs = list(zip(s_batch, r_batch))
            preds = hhem_model.predict(batch_pairs)
            consistent_prob = np.asarray(preds, dtype=float).reshape(-1)
        else:
            enc = tokenizer(s_batch, r_batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
            with torch.no_grad():
                out = hhem_model(**enc)
                logits = out.logits if hasattr(out, "logits") else out[0]
                consistent_prob = torch.sigmoid(logits.squeeze(-1)).cpu().numpy()
        hhem_scores.extend((1.0 - consistent_prob).tolist())

    hhem_hall = np.array(hhem_scores, dtype=float)
    rows.append(eval_detector("Vectara HHEM (1 - P_consistent, capped)", y_test, hhem_hall, time.time() - t0))
    print("Done: Vectara HHEM")
except Exception as e:
    print("Vectara HHEM skipped due to model wrapper error:", repr(e))

results_df = pd.DataFrame(rows)
results_df.to_csv("outputs/benchmark_results.csv", index=False)
print("\n=== BENCHMARK RESULTS ===")
print(results_df.to_string(index=False))

Using device: cpu


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Done: MiniLM


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

[transformers] You are using a model of type `HHEMv2Config` to instantiate a model of type `HHEMv2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Done: DeBERTa NLI
Vectara HHEM skipped due to model wrapper error: ValueError("Couldn't instantiate the backend tokenizer from one of: \n(1) a `tokenizers` library serialization file, \n(2) a slow tokenizer instance to convert or \n(3) an equivalent slow tokenizer class to instantiate and convert. \nYou need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.")

=== BENCHMARK RESULTS ===
                              model  n_test  roc_auc  pr_auc  best_f1  precision  recall  thresh    sec
               MiniLM (1 - cos_sim)     250   0.5982  0.3015   0.4286     0.3020  0.7377  0.1935 138.15
DeBERTa-v3-small NLI (1 - P_entail)     250   0.5994  0.3600   0.4115     0.2747  0.8197  0.9949 395.13
